# NTL training unit — LoRA24, one file for Kaggle and Colab

Trains the strongest single-member recipe in this competition (a LoRA on the first layers of a
Qwen3 backbone, pooled at 8 depths, with a plain head and a same-prompt *centred* head) and writes
exactly the artifact the inference notebook already reads.

**Run it on either platform.** `PLATFORM` below, or the `NTL_PLATFORM` environment variable:

| | Kaggle | Colab |
|---|---|---|
| set | `NTL_PLATFORM=kaggle` (default) | `NTL_PLATFORM=colab` |
| data | already mounted at `/kaggle/input` | downloaded with the Kaggle CLI into `/content/data` |
| base model | the attached Kaggle model | `Qwen/Qwen3-8B-Base` from the Hub |
| output | `/kaggle/working/unit` (comes back with the kernel output) | `/content/unit`, then `kaggle datasets create` |

**Unit 0 is a smoke run**: 12 layers, 8k pool passages, 300 stage-1 steps, a 90-minute time box.
Its job is to (a) prove the artifact loads in the inference notebook, (b) measure passages/second so
the later units can be sized from data instead of guesses, and (c) print an honest prompt-grouped
holdout score. Scale `UNIT` up (keep 24, layers `[2,4,6,8,12,16,20,24]`, `pool_rows` 75783,
`stage1_steps` = one full epoch) for a real member.

**Artifact contract** — the same three things every published checkpoint has:

    adapter/            adapter_config.json + adapter_model.safetensors (PEFT)
    heads.safetensors   norms.{i}.weight/bias, plain.weight/bias, centered.weight/bias
    meta.json           {"labels": [...], "cfg": {"keep", "layers", "max_len", "eos_id"}}

The `eos_id` matters: Kaggle's Qwen3 bases ship the chat tokenizer's end token (151645) while these
checkpoints are trained with 151643, and reading a passage with the wrong end token silently changes
about 14% of the top predictions.

In [ ]:
"""LoRA24 training unit — runs on Kaggle or Colab from one file.

Set NTL_PLATFORM=kaggle (default) or colab, or edit PLATFORM below.
Artifact contract (what the inference notebook reads):
    adapter/            PEFT adapter (adapter_config.json + adapter_model.safetensors)
    heads.safetensors   norms.{i}.weight/bias, plain.weight/bias, centered.weight/bias
    meta.json           {"labels": [...], "cfg": {"keep", "layers", "max_len", "eos_id"}}
"""
import gc, glob, json, os, subprocess, sys, time

PLATFORM = os.environ.get("NTL_PLATFORM", "kaggle").lower()
INPUT_DIR = "/kaggle/input"
OUT_DIR = "/kaggle/working/unit" if PLATFORM == "kaggle" else "/content/unit"
DRIVE_CKPT = "/content/drive/MyDrive/ntl_ckpt" if PLATFORM == "colab" else None

# ---- the unit's config. Unit 0 is a smoke + throughput run; scale these up later ----
UNIT = dict(
    name="unit0",
    base_hf="Qwen/Qwen3-8B-Base",
    keep=12,                              # decoder layers kept (24 is the full recipe)
    layers=[2, 4, 6, 8, 12],              # depths pooled into the feature vector
    max_len=384,
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    pool_rows=8000,                       # stage-1 rows sampled from the pool
    bs=4, lr=1e-4, head_lr=1e-3, wd=0.01,
    stage1_steps=300,                     # stage 1 = public pool
    stage2_epochs=1,                      # stage 2 = host train passages
    holdout_frac=0.2,                     # prompt-grouped holdout for the score print
    max_minutes=90,                       # hard time box: stop cleanly and still save
    ckpt_every=100,
)

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
LABEL2I = {l: i for i, l in enumerate(LABELS)}
K = len(LABELS)

T0 = time.time()
os.makedirs(OUT_DIR, exist_ok=True)
print(f"platform={PLATFORM} unit={UNIT['name']} out={OUT_DIR}", flush=True)

if PLATFORM == "colab":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.56", "safetensors", "accelerate", "kaggle"], check=False)

import numpy as np
import torch
from torch import nn
from safetensors.torch import load_file, save_file

torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
import transformers
try:
    import peft
    PEFT_V = peft.__version__
except Exception as e:
    PEFT_V = f"MISSING ({e})"
print(f"torch={torch.__version__} transformers={transformers.__version__} peft={PEFT_V}", flush=True)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEV} gpus={torch.cuda.device_count()} "
      f"gpu={torch.cuda.get_device_name(0) if DEV == 'cuda' else '-'}", flush=True)

## 1. Data: host passages with a prompt-grouped holdout, plus the public pool

In [ ]:
"""Data: host train passages + the public pool, on either platform."""

def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {INPUT_DIR}")
    return hits[0]


if PLATFORM == "kaggle":
    import pandas as pd
    train = pd.read_csv(find("train.csv"))
    pool = pd.read_parquet(find("pool_v4.parquet"))
    BASE_DIR = None   # resolved from the attached Kaggle model below
else:
    import pandas as pd
    os.makedirs("/content/data", exist_ok=True)
    if not os.path.exists("/content/data/train.csv"):
        subprocess.run(["kaggle", "competitions", "download", "-c", "name-that-llm",
                        "-p", "/content/data"], check=False)
        for z in glob.glob("/content/data/*.zip"):
            subprocess.run(["unzip", "-o", z, "-d", "/content/data"], check=False)
    if not os.path.exists("/content/data/pool_v4.parquet"):
        subprocess.run(["kaggle", "datasets", "download", "-d", "nishantdahal/ntl-handwriting-blend",
                        "-p", "/content/data", "--unzip"], check=False)
    train = pd.read_csv("/content/data/train.csv")
    pool = pd.read_parquet(glob.glob("/content/data/**/pool_v4.parquet", recursive=True)[0])
    BASE_DIR = UNIT["base_hf"]

pool = pool.dropna(subset=["text", "model"]).copy()
pool["text"] = pool["text"].astype(str)
pool["model"] = pool["model"].astype(str)
pool = pool[pool["model"].isin(LABELS)].copy()
train = train.dropna(subset=["text", "model"]).copy()
train["text"] = train["text"].astype(str)

# host data: prompt-grouped holdout, so the printed score is honest
from sklearn.model_selection import GroupShuffleSplit
groups = train["prompt_id"].astype(str).values if "prompt_id" in train else train.index.astype(str).values
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=UNIT["holdout_frac"], random_state=42)
                      .split(train, groups=groups))
host_tr, host_va = train.iloc[tr_idx], train.iloc[va_idx]
print(f"host train {len(host_tr):,} / holdout {len(host_va):,} passages "
      f"({host_tr.prompt_id.nunique()} / {host_va.prompt_id.nunique()} prompts)", flush=True)

# stage-1 sample from the pool, capped for the unit
stage1 = pool.sample(min(UNIT["pool_rows"], len(pool)), random_state=0)
print(f"stage1 pool sample {len(stage1):,} of {len(pool):,}", flush=True)

y_tr = host_tr["model"].map(LABEL2I).values
y_va = host_va["model"].map(LABEL2I).values
y_s1 = stage1["model"].map(LABEL2I).values

## 2. Trunk, LoRA, heads

In [ ]:
"""Trunk + a hand-rolled LoRA + the two heads.

No peft: Kaggle's image ships a torchao that peft refuses to import, and the training kernel has
no internet to fix it. A LoRA is two small matrices per adapted layer, so it is written here
directly — and the adapter file it writes is the same format the inference notebook merges.
"""
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    """Kaggle: the attached model whose config.json matches. Colab: the HF id."""
    if BASE_DIR:
        return BASE_DIR
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


class LoRALinear(nn.Module):
    """out = base(x) + dropout(x) A^T B^T * alpha / r, with B initialised to zero."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.drop(x) @ self.A.T @ self.B.T * self.scale


def inject_lora(model, targets, r, alpha, dropout):
    hits = 0
    for name, module in list(model.named_modules()):
        short = name.rsplit(".", 1)[-1]
        if isinstance(module, nn.Linear) and short in targets:
            parent = model.get_submodule(name.rsplit(".", 1)[0]) if "." in name else model
            setattr(parent, short, LoRALinear(module, r, alpha, dropout))
            hits += 1
    print(f"LoRA injected into {hits} linear layers (r={r}, alpha={alpha})", flush=True)
    if hits == 0:
        raise RuntimeError("no target modules matched — check the module names of this backbone")
    return model


BASE_SPEC = dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)   # Qwen3-8B
base_dir = find_base(BASE_SPEC)

mc = AutoConfig.from_pretrained(base_dir)
mc.num_hidden_layers, mc.use_cache = UNIT["keep"], False
if getattr(mc, "layer_types", None) is not None:
    mc.layer_types = mc.layer_types[:UNIT["keep"]]
H = mc.hidden_size

dtype = torch.bfloat16 if DEV == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
trunk = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to(DEV)
trunk.norm = nn.Identity()          # the published checkpoints were trained without the final norm
for p in trunk.parameters():
    p.requires_grad = False         # only the LoRA matrices and the heads learn
model = inject_lora(trunk, ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                    UNIT["lora_r"], UNIT["lora_alpha"], UNIT["lora_dropout"])
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable LoRA parameters: {n_train:,}", flush=True)

tok = AutoTokenizer.from_pretrained(base_dir)
tok.padding_side = "right"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
# The end token the checkpoint must be read with. Kaggle's Qwen3-8B-Base ships the chat
# tokenizer's <|im_end|> (151645); the public checkpoints were trained with 151643.
EOS_ID = 151643
print(f"end token {EOS_ID}, this tokenizer's is {tok.eos_token_id}", flush=True)

norms = nn.ModuleList([nn.LayerNorm(H) for _ in UNIT["layers"]]).to(DEV).float()
plain = nn.Linear(H * len(UNIT["layers"]), K).to(DEV)
centered = nn.Linear(H * len(UNIT["layers"]), K).to(DEV)


@torch.no_grad()
def pool_features(texts, bs=UNIT["bs"]):
    """Mean-pooled hidden states at UNIT['layers'], one LayerNorm per depth."""
    model.eval()
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=UNIT["max_len"])["input_ids"]
    ids = [x + [EOS_ID] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")
    Z = np.zeros((len(ids), H * len(UNIT["layers"])), np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(DEV)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        m = enc["attention_mask"].unsqueeze(-1).float()
        m[:, 0] = 0                      # the first token is an attention sink
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(UNIT["layers"], norms)], -1).cpu().numpy()
    return Z

## 3. Training: pool first, then the host passages with the centred head

In [ ]:
"""The training loop: plain batch + a same-prompt batch for the centred head, time-boxed."""

opt = torch.optim.AdamW([
    {"params": [p for p in model.parameters() if p.requires_grad], "lr": UNIT["lr"]},
    {"params": list(norms.parameters()) + list(plain.parameters()) + list(centered.parameters()),
     "lr": UNIT["head_lr"]},
], weight_decay=UNIT["wd"])
ce = nn.CrossEntropyLoss()

stats = dict(steps=0, passages=0, stage1_steps=0, stage2_steps=0, stopped_early=False)


def batches(df, y, bs, rng, steps=None):
    """Random batches, yielding (texts, labels, prompt ids)."""
    idx = rng.permutation(len(df))
    out = []
    for s in range(0, len(idx), bs):
        take = idx[s:s + bs]
        out.append((df.text.iloc[take].tolist(), y[take], df.prompt_id.astype(str).iloc[take].tolist()
                    if "prompt_id" in df else [str(i) for i in take]))
        if steps and len(out) >= steps:
            break
    return out


def same_prompt_batch(df, y, rng, n_prompts=2):
    """Rows that share a prompt, so the centred head sees a real topic mean to subtract."""
    if "prompt_id" not in df:
        return None
    pids = rng.choice(df.prompt_id.astype(str).unique(), size=min(n_prompts, df.prompt_id.nunique()), replace=False)
    parts = [df[df.prompt_id.astype(str) == p] for p in pids]
    sub = pd.concat(parts).sample(frac=1.0, random_state=int(rng.integers(1 << 30)))
    return sub.text.tolist(), sub["model"].map(LABEL2I).values, sub.prompt_id.astype(str).tolist()


def train_step(texts, labels, pids, want_centered=True):
    model.train()
    x = tok(texts, add_special_tokens=False, truncation=True, max_length=UNIT["max_len"],
            padding=True, return_tensors="pt").to(DEV)
    input_ids = x["input_ids"]
    eos = torch.full((input_ids.shape[0], 1), EOS_ID, device=DEV)
    input_ids = torch.cat([input_ids, eos], 1)
    att = torch.cat([x["attention_mask"], torch.ones_like(eos)], 1)
    hs = model(input_ids=input_ids, attention_mask=att, output_hidden_states=True,
               use_cache=False).hidden_states
    m = att.unsqueeze(-1).float()
    m[:, 0] = 0
    den = m.sum(1).clamp(min=1)
    Z = torch.cat([nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(UNIT["layers"], norms)], -1)
    yb = torch.tensor(labels, device=DEV)
    loss = ce(plain(Z), yb)
    if want_centered:
        pids_t = np.array(pids)
        Zc = Z.clone()
        for p in np.unique(pids_t):
            rows = np.flatnonzero(pids_t == p)
            if len(rows) >= 2:
                Zc[rows] = Zc[rows] - Zc[rows].mean(0, keepdim=True)
        loss = loss + ce(centered(Zc), yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    opt.step()
    return float(loss.detach())


rng = np.random.default_rng(0)
t_start = time.time()


def over_budget():
    return (time.time() - T0) > UNIT["max_minutes"] * 60


# ---- stage 1: the public pool ----
s1 = batches(stage1, y_s1, UNIT["bs"], rng, steps=UNIT["stage1_steps"])
for i, (texts, labels, pids) in enumerate(s1, 1):
    if over_budget():
        stats["stopped_early"] = True
        print(f"time box hit during stage 1 at step {i}", flush=True)
        break
    loss = train_step(texts, labels, pids, want_centered=False)
    stats.update(steps=stats["steps"] + 1, stage1_steps=stats["stage1_steps"] + 1,
                 passages=stats["passages"] + len(texts))
    if i % 25 == 0 or i == len(s1):
        rate = stats["passages"] / max(time.time() - t_start, 1e-9)
        print(f"stage1 step {i}/{len(s1)} loss {loss:.4f} | {rate:.1f} passages/s | "
              f"{time.time() - T0:.0f}s", flush=True)

# ---- stage 2: the host passages, with the centred head ----
for ep in range(UNIT["stage2_epochs"]):
    if over_budget():
        stats["stopped_early"] = True
        break
    for i, (texts, labels, pids) in enumerate(batches(host_tr, y_tr, UNIT["bs"], rng), 1):
        if over_budget():
            stats["stopped_early"] = True
            print("time box hit during stage 2", flush=True)
            break
        train_step(texts, labels, pids, want_centered=False)
        sp = same_prompt_batch(host_tr, y_tr, rng)
        if sp:
            train_step(*sp, want_centered=True)
        stats.update(steps=stats["steps"] + 1, stage2_steps=stats["stage2_steps"] + 1,
                     passages=stats["passages"] + len(texts))
        if i % 25 == 0:
            print(f"stage2 epoch {ep + 1} step {i}/{len(host_tr) // UNIT['bs']} | "
                  f"{time.time() - T0:.0f}s", flush=True)

rate = stats["passages"] / max(time.time() - t_start, 1e-9)
stats["passages_per_s"] = rate
print(f"trained {stats['steps']} steps on {stats['passages']:,} passages at {rate:.1f} passages/s", flush=True)

## 4. Honest holdout score, then the artifacts

In [ ]:
"""Score on the prompt-grouped holdout, then write the artifact contract."""


@torch.no_grad()
def evaluate(df, y):
    Z = pool_features(df.text.tolist(), bs=max(UNIT["bs"], 8))
    logits = torch.tensor(Z, device=DEV) @ plain.weight.T + plain.bias
    pred = logits.argmax(1).cpu().numpy()
    rec = [(pred[y == i] == i).mean() if (y == i).any() else float("nan") for i in range(K)]
    ba = float(np.nanmean(rec))
    return ba, {LABELS[i]: (None if np.isnan(r) else round(float(r), 4)) for i, r in enumerate(rec)}


t_eval = time.time()
ba, per_class = evaluate(host_va, y_va)
stats["holdout_ba"] = ba
stats["holdout_per_class"] = per_class
print(f"\nholdout balanced accuracy (plain head, {len(host_va):,} passages): {ba:.4f}  "
      f"({time.time() - t_eval:.0f}s)", flush=True)

# ---- artifacts: exactly what the inference notebook merges ----
adapter_dir = os.path.join(OUT_DIR, "adapter")
os.makedirs(adapter_dir, exist_ok=True)
adapter = {}
for name, module in model.named_modules():
    if isinstance(module, LoRALinear):
        short = name[name.index("layers."):]          # layers.N.self_attn.q_proj
        adapter[f"{short}.lora_A.weight"] = module.A.detach().float().cpu()
        adapter[f"{short}.lora_B.weight"] = module.B.detach().float().cpu()
save_file(adapter, os.path.join(adapter_dir, "adapter_model.safetensors"))
with open(os.path.join(adapter_dir, "adapter_config.json"), "w") as f:
    json.dump({"peft_type": "LORA", "r": UNIT["lora_r"], "lora_alpha": UNIT["lora_alpha"],
               "lora_dropout": UNIT["lora_dropout"], "task_type": None,
               "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"]}, f, indent=2)

heads = {}
for i, nm in enumerate(norms):
    heads[f"norms.{i}.weight"] = nm.weight.detach().float().cpu()
    heads[f"norms.{i}.bias"] = nm.bias.detach().float().cpu()
heads["plain.weight"] = plain.weight.detach().float().cpu()
heads["plain.bias"] = plain.bias.detach().float().cpu()
heads["centered.weight"] = centered.weight.detach().float().cpu()
heads["centered.bias"] = centered.bias.detach().float().cpu()
save_file(heads, os.path.join(OUT_DIR, "heads.safetensors"))

meta = {"labels": LABELS,
        "cfg": {"keep": UNIT["keep"], "layers": UNIT["layers"], "max_len": UNIT["max_len"],
                "eos_id": EOS_ID, "base": UNIT["base_hf"], "unit": UNIT["name"]},
        "holdout_ba": ba, "stats": stats, "platform": PLATFORM,
        "note": "trained by notebooks/train-lora24.ipynb (hand-rolled LoRA, no peft)"}
with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
with open(os.path.join(OUT_DIR, "COMPLETE.json"), "w") as f:
    json.dump({"unit": UNIT["name"], "steps": stats["steps"], "holdout_ba": ba,
               "passages_per_s": stats["passages_per_s"],
               "minutes": round((time.time() - T0) / 60, 1)}, f, indent=2)

# ---- round-trip check: can the inference notebook read what we just wrote? ----
_chk = load_file(os.path.join(OUT_DIR, "heads.safetensors"))
_m = json.load(open(os.path.join(OUT_DIR, "meta.json")))
_acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
_ad = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
assert _m["labels"] == LABELS and _m["cfg"]["layers"] == UNIT["layers"]
assert all(f"norms.{i}.weight" in _chk for i in range(len(UNIT["layers"])))
assert _chk["plain.weight"].shape == (K, H * len(UNIT["layers"]))
assert _chk["centered.weight"].shape == _chk["plain.weight"].shape
assert _acfg["r"] > 0 and _acfg["lora_alpha"] > 0
assert any(".lora_A." in k for k in _ad) and any(".lora_B." in k for k in _ad)
_k = next(k for k in _ad if ".lora_A." in k)
assert "layers." in _k, "adapter keys must carry the layers.N.module path the merge slices on"
print(f"artifact contract check: OK ({len(_ad)} adapter tensors, {len(_chk)} head tensors)", flush=True)
print(json.dumps({"unit": UNIT["name"], "holdout_ba": round(ba, 4),
                  "passages_per_s": round(stats["passages_per_s"], 1),
                  "minutes": round((time.time() - T0) / 60, 1)}, indent=2), flush=True)

## 5. Publish (Colab only — on Kaggle the output is the artifact)

In [ ]:
"""Colab only: publish the artifact to Kaggle as a dataset, so the inference notebook can attach it."""
if PLATFORM == "colab":
    ds = {"title": f"NTL {UNIT['name']} lora24 adapter",
          "id": f"{os.environ.get('KAGGLE_USERNAME', 'janaiworkspace')}/ntl-{UNIT['name']}-lora24",
          "licenses": [{"name": "CC0-1.0"}]}
    with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
        json.dump(ds, f, indent=2)
    r = subprocess.run(["kaggle", "datasets", "create", "-p", OUT_DIR], capture_output=True, text=True)
    print(r.stdout or r.stderr, flush=True)
    print("dataset slug:", ds["id"], flush=True)
else:
    print("Kaggle run: the artifact is already in /kaggle/working/unit and comes back with the kernel output",
          flush=True)
    print("Next step in CI: kaggle datasets create -p kaggle-output/unit", flush=True)